In [2]:
import pandas as pd
df = pd.read_csv('df_ols.csv',sep=',')
df

,user_id,speed,conversion,device_type,device_age,region,query_complexity,results_count
0,245487,2.750632,0,ios,new,bukhara,5,76
1,147916,2.848162,1,android,new,bukhara,3,78
2,740659,3.534081,0,android,old,bukhara,2,94
3,854907,1.759467,0,android,new,tashkent,3,45
4,819644,1.311305,0,android,new,tashkent,2,35
...,...,...,...,...,...,...,...,...
9995,942015,0.641197,0,ios,new,tashkent,2,14
9996,509010,0.787542,0,ios,new,tashkent,3,22
9997,948219,3.551813,0,android,old,bukhara,2,86
9998,956892,0.992976,0,ios,new,tashkent,3,62


In [3]:
df['speed'].corr(df['conversion'])

np.float64(-0.0303222482168239)

In [4]:
df.groupby('device_type')['conversion'].mean()

device_type
android    0.114407
ios        0.157018
Name: conversion, dtype: float64

In [5]:
df['results_count'].corr(df['conversion'])

np.float64(0.032848981492270245)

In [6]:
df['results_count'].corr(df['speed'])

np.float64(0.742553613491573)

In [7]:
# разбиваем на квартили по speed
df['speed_quartile'] = pd.qcut(df['speed'], q=4)
print(df.groupby('speed_quartile')['conversion'].mean())

speed_quartile
(-0.182, 1.549]    0.1480
(1.549, 2.095]     0.1376
(2.095, 2.604]     0.1324
(2.604, 4.182]     0.1256
Name: conversion, dtype: float64


/var/folders/6r/hytlw2sn1fs_9jzjy_lsxtg80000gn/T/ipykernel_33976/3846934702.py:3: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(df.groupby('speed_quartile')['conversion'].mean())


In [8]:
print(df['speed'].corr(df['conversion']))

-0.0303222482168239


In [9]:
print(df.groupby('device_type')['conversion'].mean())

device_type
android    0.114407
ios        0.157018
Name: conversion, dtype: float64


In [10]:
# разбиваем на квартили по speed
df['speed_quartile'] = pd.qcut(df['speed'], q=4)
print(df.groupby('speed_quartile')['conversion'].mean())

speed_quartile
(-0.182, 1.549]    0.1480
(1.549, 2.095]     0.1376
(2.095, 2.604]     0.1324
(2.604, 4.182]     0.1256
Name: conversion, dtype: float64


/var/folders/6r/hytlw2sn1fs_9jzjy_lsxtg80000gn/T/ipykernel_33976/3846934702.py:3: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(df.groupby('speed_quartile')['conversion'].mean())


In [11]:
print('results_count & speed:', df['results_count'].corr(df['speed']))
print('results_count & conversion:', df['results_count'].corr(df['conversion']))

results_count & speed: 0.742553613491573
results_count & conversion: 0.032848981492270245


In [12]:
import statsmodels.formula.api as smf

# Шаг 1: регрессия speed на все конфаундеры
debiase_model = smf.ols(
    'speed ~ device_type + device_age + region + query_complexity + results_count',
    data=df
).fit()

# Получаем остатки
df['speed_resid'] = debiase_model.resid

# Корреляция остатков с query_complexity
corr = df['speed_resid'].corr(df['query_complexity'])
print(round(corr, 2))

-0.0


In [ ]:
# Шаг 2: регрессия conversion на все конфаундеры
denoise_model = smf.ols(
    'conversion ~ device_type + device_age + region + query_complexity + results_count',
    data=df
).fit()

# Получаем остатки
df['conversion_resid'] = denoise_model.resid

# Корреляция остатков conversion с остатками speed из прошлого шага
corr = df['conversion_resid'].corr(df['speed_resid'])
print(round(corr, 2))


-0.03


In [14]:
# Шаг 3: регрессия остатков conversion на остатки speed
final_model = smf.ols('conversion_resid ~ speed_resid', data=df).fit()
ate_fwl = final_model.params['speed_resid']
print('ATE через FWL:', round(ate_fwl, 2))

# Сравниваем с короткой регрессией (без конфаундеров)
short_model = smf.ols('conversion ~ speed', data=df).fit()
ate_short = short_model.params['speed']
print('ATE короткая регрессия:', round(ate_short, 2))

# Сравниваем с полной регрессией (с конфаундерами)
full_model = smf.ols(
    'conversion ~ speed + device_type + device_age + region + query_complexity + results_count',
    data=df
).fit()
ate_full = full_model.params['speed']
print('ATE полная регрессия:', round(ate_full, 2))

ATE через FWL: -0.05
ATE короткая регрессия: -0.01
ATE полная регрессия: -0.05


In [15]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

np.random.seed(42)
n = 10000

# Конфаундер: лояльность (0 до 1)
loyalty = np.random.uniform(0, 1, n)

# Логин зависит от лояльности
# чем лояльнее — тем чаще логинится
login = np.random.binomial(1, p=loyalty, size=n)

# Конверсия зависит от лояльности И логина
# истинный эффект логина = 0.05
true_ate = 0.05
conversion_prob = 0.1 + 0.4 * loyalty + true_ate * login
conversion_prob = np.clip(conversion_prob, 0, 1)
conversion = np.random.binomial(1, p=conversion_prob, size=n)

df = pd.DataFrame({
    'loyalty': loyalty,
    'login': login,
    'conversion': conversion
})

# Наивная оценка (без контроля конфаундера)
naive = smf.ols('conversion ~ login', data=df).fit()
print('Наивная оценка:', round(naive.params['login'], 3))

# Правильная оценка (с контролем лояльности)
correct = smf.ols('conversion ~ login + loyalty', data=df).fit()
print('Правильная оценка:', round(correct.params['login'], 3))

print('Истинный ATE:', true_ate)

Наивная оценка: 0.164
Правильная оценка: 0.035
Истинный ATE: 0.05
